# Practical Case Study: Predicting Match Outcomes

**Chapter 6: Regression Techniques for Soccer Analytics**

## Learning Objectives

- Apply regression techniques to a complete real-world problem
- Perform end-to-end workflow: EDA → Feature Engineering → Modeling → Evaluation
- Build and compare multiple regression models
- Make actionable predictions for match outcomes
- Interpret results in soccer context
- Understand practical considerations and limitations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


## The Problem

**Business Question:** Can we predict how many goals a team will score in their next match?

**Why it matters:**
- Pre-match tactical planning
- Betting market analysis
- Fan engagement (prediction games)
- Squad rotation decisions

**Approach:** Use historical team performance data to build a predictive model.


## Step 1: Load and Explore the Data


In [ ]:
# Simulated season data for multiple teams
np.random.seed(42)
n_matches = 200

teams = ['Arsenal', 'Chelsea', 'Liverpool', 'Man City', 'Man United', 'Tottenham']

data = {
    'Team': np.random.choice(teams, n_matches),
    'ShotsOnTarget': np.random.randint(3, 15, n_matches),
    'Possession': np.random.uniform(35, 70, n_matches),
    'PassAccuracy': np.random.uniform(70, 92, n_matches),
    'xG': np.random.uniform(0.5, 3.5, n_matches),
    'OpponentStrength': np.random.uniform(0.3, 0.9, n_matches),  # 0=weak, 1=strong
    'HomeAdvantage': np.random.choice([0, 1], n_matches),  # 0=away, 1=home
}

df = pd.DataFrame(data)

# Create realistic goals based on features
df['Goals'] = (
    df['xG'] * 0.7 + 
    df['ShotsOnTarget'] * 0.08 + 
    df['HomeAdvantage'] * 0.3 - 
    df['OpponentStrength'] * 0.5 +
    np.random.normal(0, 0.4, n_matches)
).clip(0, 6).round().astype(int)

print("Match Dataset:")
print(df.head(10))
print(f"\
Dataset shape: {df.shape}")
print(f"\
Goals distribution:")
print(df['Goals'].value_counts().sort_index())


## Step 2: Exploratory Data Analysis


In [ ]:
# Summary statistics
print("Dataset Summary:")
print(df.describe())

# Check for missing values
print(f"\
Missing values:")
print(df.isnull().sum())

# Goals by team
print(f"\
Average goals by team:")
print(df.groupby('Team')['Goals'].mean().sort_values(ascending=False))

# Home vs Away
print(f"\
Goals by venue:")
print(df.groupby('HomeAdvantage')['Goals'].agg(['mean', 'count']))
print("(0=Away, 1=Home)")


## Visualize Key Relationships


In [ ]:
# Create correlation heatmap
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Correlation matrix
numeric_cols = ['ShotsOnTarget', 'Possession', 'PassAccuracy', 'xG', 'OpponentStrength', 'HomeAdvantage', 'Goals']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0, 0])
axes[0, 0].set_title('Feature Correlation Matrix')

# 2. Goals vs xG
axes[0, 1].scatter(df['xG'], df['Goals'], alpha=0.5)
axes[0, 1].set_xlabel('Expected Goals (xG)')
axes[0, 1].set_ylabel('Actual Goals')
axes[0, 1].set_title('Goals vs. xG')
axes[0, 1].grid(True, alpha=0.3)

# 3. Goals distribution
axes[1, 0].hist(df['Goals'], bins=range(8), edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Goals Scored')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Goals')

# 4. Home vs Away
home_away_data = df.groupby('HomeAdvantage')['Goals'].mean()
axes[1, 1].bar(['Away', 'Home'], home_away_data.values, alpha=0.7, color=['coral', 'skyblue'])
axes[1, 1].set_ylabel('Average Goals')
axes[1, 1].set_title('Home Advantage Effect')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key Insights:")
print(f"- xG has strongest correlation with Goals: {corr_matrix.loc['xG', 'Goals']:.2f}")
print(f"- Home advantage adds ~{home_away_data[1] - home_away_data[0]:.2f} goals on average")
print(f"- Opponent strength negatively correlates: {corr_matrix.loc['OpponentStrength', 'Goals']:.2f}")


## Step 3: Feature Engineering

Create additional features to improve predictions.


In [ ]:
# Create interaction features
df['xG_ShotsOnTarget'] = df['xG'] * df['ShotsOnTarget']
df['HomeVsStrong'] = df['HomeAdvantage'] * (1 - df['OpponentStrength'])
df['ShotEfficiency'] = df['xG'] / (df['ShotsOnTarget'] + 1)  # Avoid division by zero

# Create team strength encoding (based on historical performance)
team_strength = df.groupby('Team')['Goals'].mean().to_dict()
df['TeamStrength'] = df['Team'].map(team_strength)

print("New features created:")
print(df[['xG_ShotsOnTarget', 'HomeVsStrong', 'ShotEfficiency', 'TeamStrength']].head())
print(f"\
Team strength encoding:")
for team, strength in sorted(team_strength.items(), key=lambda x: x[1], reverse=True):
    print(f"  {team}: {strength:.2f} goals/match")


## Step 4: Prepare Data for Modeling


In [ ]:
# Select features
feature_cols = [
    'ShotsOnTarget', 'Possession', 'PassAccuracy', 'xG', 
    'OpponentStrength', 'HomeAdvantage', 'TeamStrength',
    'xG_ShotsOnTarget', 'HomeVsStrong', 'ShotEfficiency'
]

X = df[feature_cols]
y = df['Goals']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")
print(f"\
Features: {len(feature_cols)}")
print(feature_cols)


## Step 5: Build and Compare Models

We'll compare three approaches:
1. **Linear Regression** - Simple, interpretable
2. **Poisson Regression** - Designed for count data
3. **KNN Regression** - Non-parametric, flexible


In [ ]:
# 1. Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

# 2. Poisson Regression
# Prepare data for statsmodels
X_train_poisson = X_train.copy()
X_train_poisson['Goals'] = y_train.values
formula = 'Goals ~ ' + ' + '.join(feature_cols)
poisson_model = smf.poisson(formula, data=X_train_poisson).fit()

X_test_poisson = X_test.copy()
poisson_pred = poisson_model.predict(X_test_poisson)

# 3. KNN Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsRegressor(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

print("All models trained successfully!")


## Step 6: Evaluate and Compare


In [ ]:
# Calculate metrics for all models
models_results = []

for name, predictions in [('Linear', linear_pred), ('Poisson', poisson_pred), ('KNN', knn_pred)]:
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    
    models_results.append({
        'Model': name,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae
    })

results_df = pd.DataFrame(models_results)
print("Model Performance Comparison:")
print(results_df.to_string(index=False))
print(f"\
Best model by R²: {results_df.loc[results_df['R²'].idxmax(), 'Model']}")
print(f"Best model by MAE: {results_df.loc[results_df['MAE'].idxmin(), 'Model']}")


## Visualize Model Predictions


In [ ]:
# Create prediction comparison plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, predictions) in enumerate([('Linear', linear_pred), ('Poisson', poisson_pred), ('KNN', knn_pred)]):
    axes[idx].scatter(y_test, predictions, alpha=0.6)
    axes[idx].plot([0, 6], [0, 6], 'r--', linewidth=2, label='Perfect Prediction')
    axes[idx].set_xlabel('Actual Goals')
    axes[idx].set_ylabel('Predicted Goals')
    axes[idx].set_title(f'{name} Regression')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_xlim(-0.5, 6.5)
    axes[idx].set_ylim(-0.5, 6.5)

plt.tight_layout()
plt.show()

print("Points closer to the red line = better predictions")


## Step 7: Interpret the Best Model

Let's examine the linear model coefficients for interpretability.


In [ ]:
# Get feature importance from linear model
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': linear_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("Feature Importance (Linear Regression):")
print(coef_df.to_string(index=False))
print(f"\
Intercept: {linear_model.intercept_:.3f}")

# Visualize
plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient Value')
plt.title('Feature Impact on Goals Scored')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\
Key Insights:")
print("- Green bars: Positive impact on goals")
print("- Red bars: Negative impact on goals")
print("- Larger bars: Stronger influence")


## Step 8: Make Predictions for Upcoming Matches


In [ ]:
# Create hypothetical upcoming matches
upcoming_matches = pd.DataFrame({
    'ShotsOnTarget': [10, 8, 12],
    'Possession': [55, 48, 62],
    'PassAccuracy': [85, 78, 88],
    'xG': [2.1, 1.5, 2.8],
    'OpponentStrength': [0.6, 0.8, 0.4],
    'HomeAdvantage': [1, 0, 1],
    'TeamStrength': [2.2, 1.8, 2.5]
})

# Calculate engineered features
upcoming_matches['xG_ShotsOnTarget'] = upcoming_matches['xG'] * upcoming_matches['ShotsOnTarget']
upcoming_matches['HomeVsStrong'] = upcoming_matches['HomeAdvantage'] * (1 - upcoming_matches['OpponentStrength'])
upcoming_matches['ShotEfficiency'] = upcoming_matches['xG'] / (upcoming_matches['ShotsOnTarget'] + 1)

# Make predictions
upcoming_linear = linear_model.predict(upcoming_matches[feature_cols])
upcoming_poisson = poisson_model.predict(upcoming_matches[feature_cols])
upcoming_knn = knn_model.predict(scaler.transform(upcoming_matches[feature_cols]))

print("Predictions for Upcoming Matches:")
print(f"\
{'Match':<8} {'Linear':<10} {'Poisson':<10} {'KNN':<10} {'Venue'}")
print("-" * 50)
for i in range(len(upcoming_matches)):
    venue = 'Home' if upcoming_matches.iloc[i]['HomeAdvantage'] == 1 else 'Away'
    print(f"Match {i+1:<3} {upcoming_linear[i]:<10.2f} {upcoming_poisson[i]:<10.2f} {upcoming_knn[i]:<10.2f} {venue}")

print("\
These predictions can inform:")
print("- Tactical decisions (defensive vs offensive setup)")
print("- Squad rotation (rest key players in low-scoring matches)")
print("- Fan engagement (prediction competitions)")


## Step 9: Practical Considerations

**What we learned:**

1. **Model Selection:**
   - Linear: Simple, interpretable, good baseline
   - Poisson: Theoretically sound for count data
   - KNN: Flexible but less interpretable

2. **Feature Engineering Matters:**
   - Interaction terms capture complex relationships
   - Domain knowledge improves features
   - Team strength encoding adds context

3. **Limitations:**
   - Can't predict rare events (red cards, injuries)
   - Historical data may not reflect current form
   - External factors (weather, motivation) not captured

4. **Real-World Deployment:**
   - Update model regularly with new data
   - Monitor prediction accuracy over time
   - Combine with expert judgment
   - Use prediction intervals, not just point estimates


## Summary

In this case study, we:

1. ✅ Defined a clear business problem
2. ✅ Performed comprehensive EDA
3. ✅ Engineered meaningful features
4. ✅ Built and compared multiple models
5. ✅ Evaluated performance rigorously
6. ✅ Interpreted results in soccer context
7. ✅ Made actionable predictions
8. ✅ Discussed practical considerations

## Key Takeaways

- **End-to-end workflow** is crucial for real projects
- **Feature engineering** often matters more than model choice
- **Model comparison** should be systematic
- **Interpretation** makes predictions actionable
- **Practical limitations** must be acknowledged
- **Domain knowledge** enhances every step

## Congratulations!

You've completed the core notebooks for Chapter 6! You now have a solid foundation in regression techniques for soccer analytics. The extra notebooks will dive deeper into advanced topics.


## Practice Exercises

1. **Add More Features**: Include defensive metrics, recent form, head-to-head history
2. **Time Series Split**: Use temporal cross-validation instead of random split
3. **Ensemble Model**: Combine predictions from multiple models
4. **Prediction Intervals**: Add confidence intervals to predictions
5. **Real Data**: Apply to actual match data from StatsBomb or similar sources
6. **Both Teams**: Predict goals for both teams and determine match outcome
